### 법령 데이터 수집

In [15]:
import pandas as pd
import requests
import time
import os
import re
from tqdm import tqdm  # 진행바 라이브러리 추가

# 1. 설정 정보
INPUT_FILE = "전체_법률_목록(2025-12-01).csv"
SAVE_DIR = "law_xml_files"
USER_ID = "silverbreak" 
BASE_URL = "https://www.law.go.kr/DRF/lawService.do"

# 2. 저장 폴더 생성
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "_", str(name))

def download_law_xmls():
    if not os.path.exists(INPUT_FILE):
        print(f"오류: '{INPUT_FILE}' 파일이 존재하지 않습니다.")
        return

    # [중요] 주석에 '탭 구분자'라고 적혀있으므로 sep='\t'가 필요할 수 있습니다.
    # 만약 콤마(,) 구분자라면 sep 옵션을 지우세요.
    try:
        df = pd.read_csv(INPUT_FILE, encoding='utf-8', engine='python')
    except:
        # 탭 구분이 아닐 경우를 대비해 기본값으로 재시도
        df = pd.read_csv(INPUT_FILE, encoding='utf-8', engine='python')
    
    print(f"총 {len(df)}개의 법령 목록을 읽어왔습니다. 다운로드를 시작합니다.\n")
    
    # 4. 순회하며 다운로드 (tqdm으로 감싸기)
    # tqdm(대상, desc="설명") 형태로 사용
    for idx, row in tqdm(df.iterrows(), total=len(df), unit="file"):
        try:
            mst_id = row['법령MST']
            law_name = row['법령명']
            
            safe_name = sanitize_filename(law_name)
            file_path = os.path.join(SAVE_DIR, f"{safe_name}.xml")
            
            # 이미 다운로드된 파일이 있다면 건너뛰기
            if os.path.exists(file_path):
                # 진행바에 현재 처리중인 파일명만 업데이트하고 건너뜀
                continue
            
            params = {
                "OC": USER_ID,
                "target": "eflaw",
                "MST": mst_id,
                "type": "XML"
            }
            
            response = requests.get(BASE_URL, params=params)
            
            if response.status_code == 200:
                with open(file_path, "wb") as f:
                    f.write(response.content)
            else:
                # tqdm 사용 시 print 대신 tqdm.write 사용 권장 (줄깨짐 방지)
                tqdm.write(f"[{idx+1}] 요청 실패: {law_name} (Code: {response.status_code})")
            
            time.sleep(0.1)
            
        except KeyError as e:
            tqdm.write(f"컬럼 이름 오류: {e} - CSV 파일의 컬럼명을 확인해주세요.")
            break
        except Exception as e:
            tqdm.write(f"[{idx+1}] 에러 발생: {e}")

    print("\n모든 다운로드 작업이 완료되었습니다.")

if __name__ == "__main__":
    download_law_xmls()

총 1691개의 법령 목록을 읽어왔습니다. 다운로드를 시작합니다.



100%|██████████| 1691/1691 [18:30<00:00,  1.52file/s]


모든 다운로드 작업이 완료되었습니다.


In [16]:
import pandas as pd
import xml.etree.ElementTree as ET
import os
import glob
import re

# 1. 설정 정보
INPUT_DIR = "law_xml_files"       # XML 파일이 있는 폴더
OUTPUT_DIR = "law_csv_files"      # CSV를 저장할 폴더

# 2. 저장 폴더 생성
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# 3. 태그 제거 함수 (User Logic 보완)
def remove_tag(content):
    """
    HTML 태그 및 불필요한 공백 제거
    예: <br/>, <div> 등 제거
    """
    if not content:
        return ''
    # <...> 형태의 태그 제거
    clean_text = re.sub(r'<.*?>', '', str(content))
    # 연속된 공백을 하나로, 양쪽 공백 제거
    return re.sub(r'\s+', ' ', clean_text).strip()

def parse_xml_to_csv():
    # 폴더 내 모든 xml 파일 리스트 가져오기
    xml_files = glob.glob(os.path.join(INPUT_DIR, "*.xml"))
    
    print(f"총 {len(xml_files)}개의 XML 파일 변환을 시작합니다.")
    
    for idx, xml_file_path in enumerate(xml_files):
        try:
            # 파일명에서 법령명 추출 (확장자 제거)
            filename = os.path.basename(xml_file_path)
            law_name_sanitized = os.path.splitext(filename)[0]
            
            # XML 파일 파싱
            try:
                tree = ET.parse(xml_file_path)
                root = tree.getroot()
            except ET.ParseError:
                print(f"[{idx+1}] XML 파싱 에러 (건너뜀): {law_name_sanitized}")
                continue

            all_rows = []

            # --- 사용자 제공 파싱 로직 적용 ---
            for clause in root.findall(".//조문단위"):
                조문번호_tag = clause.find('조문번호')
                조문제목_tag = clause.find('조문제목')
                조문내용_tag = clause.find('조문내용')
                
                조문정보 = {
                    '법령명': law_name_sanitized,
                    '조문번호': remove_tag(조문번호_tag.text) if 조문번호_tag is not None and 조문번호_tag.text else '',
                    '조문제목': remove_tag(조문제목_tag.text) if 조문제목_tag is not None and 조문제목_tag.text else '',
                    '조문내용': remove_tag(조문내용_tag.text) if 조문내용_tag is not None and 조문내용_tag.text else ''
                }

                # 1. 조문 자체의 내용 저장 (항/호/목이 없어도 조문 내용은 저장)
                # (옵션: 조문내용이 있을 때만 저장하려면 if 조문정보['조문내용']: 조건 유지)
                if 조문정보['조문내용'] or (not clause.findall('항')): 
                    all_rows.append({**조문정보, '항': '', '호': '', '목': ''})

                # 2. 항 순회
                for depth1 in clause.findall('항'):
                    항내용_tag = depth1.find('항내용')
                    항내용 = remove_tag(항내용_tag.text) if 항내용_tag is not None and 항내용_tag.text else ''
                    
                    # 항 내용 저장
                    if 항내용:
                        all_rows.append({**조문정보, '항': 항내용, '호': '', '목': ''})

                    # 3. 호 순회
                    for depth2 in depth1.findall('호'):
                        호번호_tag = depth2.find('호번호')
                        호내용_tag = depth2.find('호내용')
                        
                        호번호 = remove_tag(호번호_tag.text) if 호번호_tag is not None and 호번호_tag.text else ''
                        호내용 = remove_tag(호내용_tag.text) if 호내용_tag is not None and 호내용_tag.text else ''
                        
                        호_full_content = f"{호번호} {호내용}".strip()
                        
                        # 호 내용 저장
                        if 호_full_content:
                            all_rows.append({**조문정보, '항': 항내용, '호': 호_full_content, '목': ''})

                        # 4. 목 순회
                        for depth3 in depth2.findall('목'):
                            목내용_tag = depth3.find('목내용')
                            목내용 = remove_tag(목내용_tag.text) if 목내용_tag is not None and 목내용_tag.text else ''
                            
                            # 목 내용 저장
                            if 목내용:
                                all_rows.append({**조문정보, '항': 항내용, '호': 호_full_content, '목': 목내용})
            
            # --- 저장 로직 ---
            if all_rows:
                save_path = os.path.join(OUTPUT_DIR, f"{law_name_sanitized}.csv")
                df = pd.DataFrame(all_rows)
                
                # 탭 구분자 사용 (\t)
                df.to_csv(save_path, index=False, encoding='utf-8-sig', sep='\t')
            
            # 진행상황 출력 (100개 단위)
            if (idx + 1) % 100 == 0:
                print(f"[{idx+1}/{len(xml_files)}] 변환 진행 중...")

        except Exception as e:
            print(f"[{idx+1}] 에러 발생 ({xml_file_path}): {e}")

    print("\n모든 변환 작업이 완료되었습니다.")

if __name__ == "__main__":
    parse_xml_to_csv()

총 1691개의 XML 파일 변환을 시작합니다.
[100/1691] 변환 진행 중...
[200/1691] 변환 진행 중...
[300/1691] 변환 진행 중...
[400/1691] 변환 진행 중...
[500/1691] 변환 진행 중...
[600/1691] 변환 진행 중...
[700/1691] 변환 진행 중...
[800/1691] 변환 진행 중...
[900/1691] 변환 진행 중...
[1000/1691] 변환 진행 중...
[1100/1691] 변환 진행 중...
[1200/1691] 변환 진행 중...
[1300/1691] 변환 진행 중...
[1400/1691] 변환 진행 중...
[1500/1691] 변환 진행 중...
[1600/1691] 변환 진행 중...

모든 변환 작업이 완료되었습니다.


### 클린징

In [17]:
import pandas as pd
import os

# 1. 설정 정보
# 앞 단계에서 생성한 폴더명
SOURCE_DIR = 'law_csv_files' 
# 결과가 저장될 폴더명
TARGET_DIR = 'law_csv_files_cleaned'

def remove_duplicate_cell_contents(df, columns):
    """
    지정된 컬럼들을 순회하며, 바로 윗 행과 값이 동일하면 빈칸('')으로 변경
    (계층 구조 시각화를 위한 처리)
    """
    for column in columns:
        if column not in df.columns:
            continue
            
        last_seen = None
        # 인덱스 순서대로 순회
        for idx in df.index:
            current_value = df.at[idx, column]
            
            # 이전 값과 같으면 빈칸 처리 (단, 첫 줄은 제외)
            if idx > 0 and current_value == last_seen:
                df.at[idx, column] = ''
            else:
                last_seen = current_value
    return df

def process_files_flat(source_folder, target_folder):
    # 폴더 생성
    if not os.path.exists(target_folder):
        os.makedirs(target_folder)
        print(f"폴더 생성: {target_folder}")

    # 파일 목록 가져오기
    if not os.path.exists(source_folder):
        print(f"오류: 원본 폴더 '{source_folder}'가 없습니다.")
        return

    file_list = [f for f in os.listdir(source_folder) if f.endswith('.csv')]
    print(f"총 {len(file_list)}개의 파일 처리를 시작합니다.")

    for i, file_name in enumerate(file_list):
        try:
            source_path = os.path.join(source_folder, file_name)
            target_path = os.path.join(target_folder, file_name)
            
            # 1. 파일 읽기 (Tab 구분자)
            df = pd.read_csv(source_path, encoding='utf-8-sig', sep='\t')
            
            # 2. 중복 내용 제거 (조문내용 -> 항 -> 호 -> 목 순서)
            # 주의: 데이터가 정렬되어 있다고 가정합니다.
            target_cols = ['조문내용', '항', '호', '목']
            df = remove_duplicate_cell_contents(df, target_cols)
            
            # 3. NaN 값을 빈 문자열로 치환
            df = df.fillna('')
            
            # 4. 저장
            df.to_csv(target_path, index=False, encoding='utf-8-sig', sep='\t')
            
            if (i + 1) % 100 == 0:
                print(f"[{i+1}/{len(file_list)}] 처리 완료...")

        except Exception as e:
            print(f"Error processing file {file_name}: {e}")
            continue

    print(f"\n모든 작업이 완료되었습니다. 결과 경로: {target_folder}")

# --- 실행부 ---
if __name__ == "__main__":
    process_files_flat(SOURCE_DIR, TARGET_DIR)

총 1691개의 파일 처리를 시작합니다.
[100/1691] 처리 완료...
[200/1691] 처리 완료...
[300/1691] 처리 완료...
[400/1691] 처리 완료...
[500/1691] 처리 완료...
[600/1691] 처리 완료...
[700/1691] 처리 완료...
[800/1691] 처리 완료...
[900/1691] 처리 완료...
[1000/1691] 처리 완료...
[1100/1691] 처리 완료...
[1200/1691] 처리 완료...
[1300/1691] 처리 완료...
[1400/1691] 처리 완료...
[1500/1691] 처리 완료...
[1600/1691] 처리 완료...

모든 작업이 완료되었습니다. 결과 경로: law_csv_files_cleaned


In [1]:
import pandas as pd
import os

# 1. 설정 정보
# 중요: 텍스트 분석용이라면 'cleaned'가 아닌 원본 'law_csv_files'를 사용하는 것이 좋습니다.
# 만약 이미 빈칸 처리가 된 파일을 써야 한다면, 아래 코드의 주석(ffill) 부분을 참고하세요.
SOURCE_DIR = '법률_정제_완료' 
TARGET_DIR = '법률_정제_완료_txt화'

def convert_csv_to_txt(source_folder, target_folder):
    # 폴더 생성
    if not os.path.exists(target_folder):
        os.makedirs(target_folder)
        print(f"결과 저장 폴더 생성: {target_folder}")

    if not os.path.exists(source_folder):
        print(f"오류: 원본 폴더 '{source_folder}'가 없습니다.")
        return

    file_list = [f for f in os.listdir(source_folder) if f.endswith('.csv')]
    print(f"총 {len(file_list)}개의 파일 변환을 시작합니다.")

    for i, file_name in enumerate(file_list):
        try:
            source_path = os.path.join(source_folder, file_name)
            # txt 파일명 생성 (확장자만 변경)
            target_file_name = os.path.splitext(file_name)[0] + '.txt'
            target_path = os.path.join(target_folder, target_file_name)
            
            # 1. 파일 읽기
            df = pd.read_csv(source_path, encoding='utf-8-sig', sep='\t')
            
            # 2. 필요한 컬럼만 선택
            target_cols = ['조문내용', '항', '호', '목']
            
            # 해당 컬럼이 실제로 존재하는지 확인 (없으면 생성 또는 무시)
            existing_cols = [col for col in target_cols if col in df.columns]
            
            if not existing_cols:
                print(f"경고: {file_name}에 대상 컬럼이 없습니다. 건너뜁니다.")
                continue

            # 3. 결측치(NaN) 처리
            # 텍스트로 합치기 위해 NaN을 빈 문자열 ''로 변환
            df_subset = df[existing_cols].fillna('')
            
            # [중요] 만약 입력 데이터가 이미 'cleaned(중복제거됨)' 상태라면,
            # 아래 주석을 해제하여 빈칸을 앞의 값으로 채워야(ffill) 온전한 문장이 됩니다.
            # df_subset = df_subset.replace('', pd.NA).ffill().fillna('')

            # 4. 컬럼 내용 합치기 (공백으로 구분)
            # 예: "제1조(목적) ① 이 법은..." 처럼 한 줄로 만듭니다.
            # strip()을 사용하여 불필요한 앞뒤 공백 제거
            text_lines = df_subset.apply(
                lambda row: ' '.join(row.values.astype(str)).strip(), axis=1
            )
            
            # 빈 줄 제거 (선택 사항)
            text_lines = text_lines[text_lines != '']

            # 5. txt 파일로 저장
            with open(target_path, 'w', encoding='utf-8') as f:
                # 각 행을 줄바꿈(\n)으로 구분하여 쓰기
                f.write('\n'.join(text_lines))
            
            if (i + 1) % 100 == 0:
                print(f"[{i+1}/{len(file_list)}] 변환 완료...")

        except Exception as e:
            print(f"Error processing file {file_name}: {e}")
            continue

    print(f"\n모든 작업이 완료되었습니다. 결과 경로: {target_folder}")

if __name__ == "__main__":
    convert_csv_to_txt(SOURCE_DIR, TARGET_DIR)

총 1690개의 파일 변환을 시작합니다.
[100/1690] 변환 완료...
[200/1690] 변환 완료...
[300/1690] 변환 완료...
[400/1690] 변환 완료...
[500/1690] 변환 완료...
[600/1690] 변환 완료...
[700/1690] 변환 완료...
[800/1690] 변환 완료...
[900/1690] 변환 완료...
[1000/1690] 변환 완료...
[1100/1690] 변환 완료...
[1200/1690] 변환 완료...
[1300/1690] 변환 완료...
[1400/1690] 변환 완료...
[1500/1690] 변환 완료...
[1600/1690] 변환 완료...

모든 작업이 완료되었습니다. 결과 경로: 법률_정제_완료_txt화
